# TVD minmod follow-up for linear advection
This notebook mirrors the repo's limiter sidecar: same periodic transport problem, one extra bounded scheme, and one sharper question. Can a TVD limiter keep the jump monotone without falling all the way back to first-order blur?


## Flux used in the sidecar
For positive advection speed, the update is written as `u_i^{n+1} = u_i^n - nu (F_{i+1/2} - F_{i-1/2})`, with `F_{i+1/2} = u_i + 0.5 (1-nu) phi(r_i) (u_{i+1} - u_i)` and `phi(r) = max(0, min(1, r))`.

That keeps the follow-up honest: one new scheme, not a whole new solver family.


In [ ]:
from advectionlab.analysis import study_transport
rows = study_transport(
    schemes=('upwind', 'lax-friedrichs', 'lax-wendroff', 'tvd-minmod'),
    requested_cfls=(0.4, 0.7, 0.9),
)
for row in rows:
    if row.requested_cfl == 0.9:
        print(row)


## Reading the bounded result
At CFL 0.9 the limiter lands exactly where this repo needed it to land. It does not replace Lax-Wendroff on the smooth Gaussian, but it does keep the square pulse monotone while staying far sharper than upwind.

That makes the monotone-versus-sharp tradeoff feel like a real middle lane instead of a fake all-or-nothing choice.


In [ ]:
square_rows = [row for row in rows if row.profile_key == 'square']
for row in square_rows:
    print(row.scheme_title, row.requested_cfl, row.total_variation_ratio, row.overshoot, row.undershoot)


## Adversarial check
If the limiter had only reproduced upwind blur, the sidecar would be filler. If it had beaten every other scheme in every metric, the sidecar would feel suspiciously overclaimed. The useful result is the one in between: better bounded behavior on the jump, but still not a free pass on the smooth lane.
